# Lab 4: Fed Gen AI

The lab covers diffusion and GAN generative modeling paradigms, along with example implementations in a federated learning context.

Overview of generative modeling paradigms:

<a href="https://lilianweng.github.io/posts/2021-07-11-diffusion-models/" target="_blank"><img src="https://lilianweng.github.io/posts/2021-07-11-diffusion-models/generative-overview.png" alt="Generative Model Types" style="width:50%;"></a>

## (Federated) Diffusion Models

DDPM (Denoising Diffusion Probabilistic Models) is one of the main diffusion model formulations:

<a href="https://arxiv.org/pdf/2006.11239" target="_blank"><img src="https://lilianweng.github.io/posts/2021-07-11-diffusion-models/DDPM.png" alt="DDPM Processes" style="width:50%;"></a>

Early diffusion models uses the U-Net architecture as the denoising neural network:

<a href="https://arxiv.org/pdf/1505.04597" target="_blank"><img src="https://lilianweng.github.io/posts/2021-07-11-diffusion-models/U-net.png" alt="U-Net Layers" style="width:50%;"></a>

---

We write an HFL pipeline, structured similarly to the previous labs, but for training a diffusion model.
To get a small (untrained) U-Net DDPM model, we use the Hugging Face Diffusers framework.
We choose a [butterfly image dataset](https://huggingface.co/datasets/huggan/smithsonian_butterflies_subset).

In [ ]:
# https://huggingface.co/docs/diffusers/main/en/training/overview
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
import torch
import datasets
from torchvision import transforms

from commons.utils import get_device

DEVICE = get_device()
torch.set_float32_matmul_precision("high")

IMAGE_SIZE = 128
preprocess = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)


def transform(examples):
    images = [preprocess(image.convert("RGB")) for image in examples["image"]]
    return {"images": images}


TRAIN_DATA = datasets.load_dataset(
    "huggan/smithsonian_butterflies_subset",
    split="train").select_columns(["image"])
TRAIN_DATA.set_transform(transform)

nr_clients = 4
CLIENT_SPLIT = [
    TRAIN_DATA.shard(num_shards=nr_clients, index=i)
    for i in range(nr_clients)
]

# clients and server use the same predetermined scheduler
NOISE_SCHEDULER = DDPMScheduler(num_train_timesteps=1000)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.figure import Figure
from torchvision.utils import make_grid


def plot(images: torch.Tensor | list[torch.Tensor], title="") -> Figure:
    grid = make_grid(images, padding=2, normalize=True)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(np.transpose(grid.cpu().numpy(), (1, 2, 0)))
    ax.axis("off")

    if title != "":
        ax.set_title(title)

    return fig

In [ ]:
_ = plot(TRAIN_DATA[:4]["images"], "Example real images after preprocessing")

In [ ]:
from diffusers.models.unets.unet_2d import UNet2DModel


def create_unet_model(image_size: int) -> UNet2DModel:
    return UNet2DModel(
        sample_size=image_size,
        in_channels=3,
        out_channels=3,
        layers_per_block=2,
        block_out_channels=(128, 128, 256, 256, 512, 512),
        down_block_types=(
            "DownBlock2D",
            "DownBlock2D",
            "DownBlock2D",
            "DownBlock2D",
            "AttnDownBlock2D",
            "DownBlock2D",
        ),
        up_block_types=(
            "UpBlock2D",
            "AttnUpBlock2D",
            "UpBlock2D",
            "UpBlock2D",
            "UpBlock2D",
            "UpBlock2D"
        ),
    )

In [ ]:
from typing import cast

import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader


class DiffusionWeightClient:
    def __init__(
            self, client_data: datasets.Dataset,
            lr: float, batch_size: int, nr_epochs: int,
            seed: int) -> None:
        torch.manual_seed(seed)
        self.model = create_unet_model(IMAGE_SIZE).to(DEVICE)
        # compiling makes the model faster overall, but
        # it makes the first call to the model much slower
        self.model.compile()
        self.loader_train = DataLoader(
            client_data, batch_size=batch_size, shuffle=True, drop_last=True)
        self.lr = lr
        self.nr_epochs = nr_epochs
        self.optimizer: AdamW
        self.round_loss: float

    def train_epoch(self) -> None:
        for data in self.loader_train:
            clean_images = data["images"].to(DEVICE)
            self.optimizer.zero_grad()
            # sample noise to add to the images
            noise = torch.randn(clean_images.shape).to(DEVICE)
            # sample a random timestep for each image
            timesteps = cast(
                torch.IntTensor,
                torch.randint(
                    0, NOISE_SCHEDULER.config["num_train_timesteps"],
                    (clean_images.size(0),), device=DEVICE
                )
            )
            # add noise to the clean images based on the chosen timesteps
            # later timestep -> more noise
            noisy_images = NOISE_SCHEDULER.add_noise(
                clean_images, noise, timesteps)

            # use mixed precision to make training faster and lighter on memory
            with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
                # predict the noise residual
                noise_pred = self.model(noisy_images, timesteps).sample
                # calculate MSE between predicted and actual noise
                loss = F.mse_loss(noise_pred, noise)

            loss.backward()
            self.optimizer.step()
            self.round_loss += loss.item()

    def update(self, in_params: list[torch.Tensor], seed: int) -> list[torch.Tensor]:
        with torch.no_grad():
            for client_values, server_values in zip(self.model.parameters(), in_params):
                client_values[:] = server_values

        torch.manual_seed(seed)
        self.round_loss = 0.
        self.model.train()
        # rinitialize optimizer to clear stale momentum
        self.optimizer = AdamW(params=self.model.parameters(), lr=self.lr)

        for _epoch in range(self.nr_epochs):
            self.train_epoch()

        self.round_loss /= self.nr_epochs * len(self.loader_train)

        return [x.detach() for x in self.model.parameters()]

In [ ]:
import numpy.random as npr
from diffusers.pipelines.ddpm.pipeline_ddpm import DDPMPipeline
from diffusers.pipelines.pipeline_utils import ImagePipelineOutput
from tqdm import trange


class DiffusionFedAvgServer:
    def __init__(
            self, lr: float, batch_size: int, client_subsets: list[datasets.Dataset],
            client_fraction: float, nr_local_epochs: int,
            seed: int) -> None:
        self.seed = seed
        torch.manual_seed(seed)
        self.model = create_unet_model(IMAGE_SIZE).to(DEVICE)

        self.nr_clients = len(client_subsets)
        self.client_fraction = client_fraction
        self.client_sample_counts = [len(subset) for subset in client_subsets]
        self.nr_clients_per_round = max(
            1, round(client_fraction * self.nr_clients))
        self.rng = npr.default_rng(seed)

        self.clients = [
            DiffusionWeightClient(
                subset, lr, batch_size, nr_local_epochs, seed)
            for subset in client_subsets
        ]

    def run(self, nr_rounds: int) -> None:
        for nr_round in (pbar := trange(nr_rounds, desc="Rounds", leave=True)):
            weights = [x.detach() for x in self.model.parameters()]
            indices_chosen_clients = self.rng.choice(
                self.nr_clients, self.nr_clients_per_round, replace=False)
            chosen_sum_nr_samples = sum(
                self.client_sample_counts[i] for i in indices_chosen_clients)
            chosen_adjusted_weights: list[list[torch.Tensor]] = []

            for c_i in indices_chosen_clients:
                ind = int(c_i)
                client_round_seed = self.seed + ind + 1 + nr_round * self.nr_clients_per_round
                client_weights = self.clients[ind].update(
                    weights, client_round_seed)
                frac = self.client_sample_counts[ind] / chosen_sum_nr_samples
                chosen_adjusted_weights.append([tens * frac for tens in client_weights])

            averaged_chosen_weights: list[torch.Tensor] = [
                torch.stack(x, dim=0).sum(dim=0) for x in zip(*chosen_adjusted_weights)]

            with torch.no_grad():
                zip_parameter_weight = zip(
                    self.model.parameters(), averaged_chosen_weights)
                for server_parameter, client_weight in zip_parameter_weight:
                    server_parameter[:] = client_weight.to(device=DEVICE)

            loss_sum = sum(self.clients[c_i].round_loss for c_i in indices_chosen_clients)
            pbar.set_postfix(loss_sum=loss_sum)

    def sample(
            self, nr_samples: int,
            num_inference_steps=200, sample_seed=0) -> torch.Tensor:
        # generate all images at once
        print(f"Generating {nr_samples} images...")
        self.model.eval()
        # create a pipeline for inference
        pipeline = DDPMPipeline(
            unet=self.model, scheduler=NOISE_SCHEDULER).to(DEVICE)
        pipeline.set_progress_bar_config(disable=True)
        generator = torch.Generator(device=DEVICE).manual_seed(sample_seed)
        pipeline = cast(
            ImagePipelineOutput,
            pipeline(
                batch_size=nr_samples,
                generator=generator,
                num_inference_steps=num_inference_steps,
                # use numpy arrays for the output
                output_type=None,
            )
        )
        output = torch.from_numpy(pipeline.images).permute(0,3,1,2)

        return output

Mimic centralized diffusion training:

In [ ]:
server = DiffusionFedAvgServer(
    lr=1e-4,
    batch_size=16,
    client_subsets=[TRAIN_DATA],
    client_fraction=1,
    nr_local_epochs=50,
    seed=0,
)
server.run(nr_rounds=1)
_ = plot(server.sample(16))

Train federated diffusion:

In [ ]:
server = DiffusionFedAvgServer(
    lr=1e-4,
    batch_size=16,
    client_subsets=CLIENT_SPLIT,
    client_fraction=0.5,
    nr_local_epochs=1,
    seed=0,
)
server.run(nr_rounds=100)
_ = plot(server.sample(16))

## (Federated) GANs

Generative adversarial networks are a generative architecture that jointly trains a generator and a discriminator model:

<a href="https://developer.ibm.com/articles/generative-adversarial-networks-explained/" target="_blank"><img src="https://developer.ibm.com/developer/default/articles/generative-adversarial-networks-explained/images/GANs.jpg" alt="GAN Architecture" style="width:50%;"></a>

The generator's goal is to synthesize data that fools the discriminator, which tries to classify input samples as real or synthetic.

In [ ]:
import torch.nn as nn

# https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=128, nc=3) -> None:
        super(Generator, self).__init__()
        self.nz = nz
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 16),
            nn.ReLU(True),
            # state size. (ngf*16) x 4 x 4
            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 8 x 8
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 16 x 16
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 32 x 32
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. (ngf) x 64 x 64
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # state size. (nc) x 128 x 128
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=128) -> None:
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # input is (nc) x 128 x 128
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf) x 64 x 64
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*2) x 32 x 32
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*4) x 16 x 16
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*8) x 8 x 8
            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*16) x 4 x 4
            nn.Conv2d(ndf * 16, 1, 4, 1, 0, bias=False)
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
def weights_init(m: nn.Module) -> None:
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
from torch.optim import Adam


class GanFedAvgClient:
    def __init__(
            self, client_data: datasets.Dataset,
            lr: float, batch_size: int, nr_epochs: int,
            beta1: float, seed: int) -> None:
        torch.manual_seed(seed)
        self.gen = Generator().to(DEVICE)
        self.discr = Discriminator().to(DEVICE)
        self.gen.compile()
        self.discr.compile()
        self.loader_train = DataLoader(
            client_data, batch_size=batch_size, shuffle=True, drop_last=True)
        self.lr = lr
        self.nr_epochs = nr_epochs
        self.beta1 = beta1
        self.opt_gen: Adam
        self.opt_discr: Adam
        self.criterion = F.binary_cross_entropy_with_logits
        self.round_loss_gen: float
        self.round_loss_discr: float

    def train_epoch(self) -> None:
        for i, data in enumerate(self.loader_train):
            ############################
            # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
            ###########################
            ## Train with all-real batch
            self.discr.zero_grad()
            # format batch
            real_cpu = data["images"].to(DEVICE)
            b_size = real_cpu.size(0)
            # apply label smoothing for better learning
            label = torch.empty(b_size, device=DEVICE).uniform_(0.9, 1.0)
            # forward pass real batch through D
            output = self.discr(real_cpu).view(-1)
            # calculate loss on all-real batch
            err_discr_real = self.criterion(output, label)
            # calculate gradients for D in backward pass
            err_discr_real.backward()
            # D_x = output.mean().item()

            ## Train with all-fake batch
            # generate batch of latent vectors
            noise = torch.randn(b_size, self.gen.nz, 1, 1, device=DEVICE)
            # generate fake image batch with G
            fake = self.gen(noise)
            label.fill_(0.)
            # classify all fake batch with D
            output = self.discr(fake.detach()).view(-1)
            # calculate D's loss on the all-fake batch
            err_discr_fake = self.criterion(output, label)
            # calculate the gradients for this batch, accumulated (summed) with previous gradients
            err_discr_fake.backward()
            # update D based on accumulated error over the fake and the real batches
            self.opt_discr.step()
            self.round_loss_gen += (err_discr_real + err_discr_fake).item()

            ############################
            # (2) Update G network: maximize log(D(G(z)))
            ###########################
            self.gen.zero_grad()
            label.fill_(1.)  # fake labels are real for generator cost
            # since we just updated D, perform another forward pass of all-fake batch through D
            output = self.discr(fake).view(-1)
            # calculate G's loss based on this output
            err_gen = self.criterion(output, label)
            # calculate gradients for G
            err_gen.backward()
            # update G
            self.opt_gen.step()
            self.round_loss_discr += err_gen.item()

    def update(
            self, gen_state_dict: dict[str, torch.Tensor],
            discr_state_dict: dict[str, torch.Tensor],
            seed: int) -> tuple[dict[str, torch.Tensor], dict[str, torch.Tensor]]:
        # use `state_dict` instead of `parameters`` to include batch norm buffers
        self.gen.load_state_dict(gen_state_dict)
        self.discr.load_state_dict(discr_state_dict)
        torch.manual_seed(seed)
        self.round_loss_gen = 0.
        self.round_loss_discr = 0.
        self.gen.train()
        self.discr.train()
        self.opt_gen = Adam(self.gen.parameters(), lr=self.lr, betas=(self.beta1, 0.999))
        self.opt_discr = Adam(self.discr.parameters(), lr=self.lr, betas=(self.beta1, 0.999))

        for _epoch in range(self.nr_epochs):
            self.train_epoch()

        self.round_loss_gen /= self.nr_epochs * len(self.loader_train)
        self.round_loss_discr /= self.nr_epochs * len(self.loader_train)

        return (self.gen.state_dict(), self.discr.state_dict())

In [ ]:
class GanFedAvgServer:
    def __init__(
            self, lr: float, batch_size: int, client_subsets: list[datasets.Dataset],
            client_fraction: float, nr_local_epochs: int,
            beta1: float, seed: int) -> None:
        self.seed = seed
        self.gen = Generator().to(DEVICE).apply(weights_init)
        self.discr = Discriminator().to(DEVICE).apply(weights_init)

        self.nr_clients = len(client_subsets)
        self.client_fraction = client_fraction
        self.client_sample_counts = [len(subset) for subset in client_subsets]
        self.nr_clients_per_round = max(
            1, round(client_fraction * self.nr_clients))
        self.rng = npr.default_rng(seed)

        self.clients = [
            GanFedAvgClient(subset, lr, batch_size, nr_local_epochs, beta1, seed)
            for subset in client_subsets
        ]

    def run(self, nr_rounds: int) -> None:
        for nr_round in (pbar := trange(nr_rounds, desc="Rounds", leave=True)):
            gen_state_dict = self.gen.state_dict()
            discr_state_dict = self.discr.state_dict()
            indices_chosen_clients = self.rng.choice(
                self.nr_clients, self.nr_clients_per_round, replace=False)
            chosen_sum_nr_samples = sum(
                self.client_sample_counts[i] for i in indices_chosen_clients)
            gen_aggr_dict = {
                k: torch.zeros_like(v) for k, v in gen_state_dict.items()}
            discr_aggr_dict = {
                k: torch.zeros_like(v) for k, v in discr_state_dict.items()}

            for c_i in indices_chosen_clients:
                ind = int(c_i)
                client_round_seed = self.seed + ind + 1 + nr_round * self.nr_clients_per_round
                client_gen_dict, client_discr_dict = self.clients[ind].update(
                    gen_state_dict, discr_state_dict, client_round_seed)
                frac = self.client_sample_counts[ind] / chosen_sum_nr_samples

                for k, v in client_gen_dict.items():
                    # BatchNorm2d.num_batches_tracked has dtype long, not float
                    # thus, the explicit cast helps avoid errors
                    gen_aggr_dict[k] += (v * frac).to(v.dtype)

                for k, v in client_discr_dict.items():
                    discr_aggr_dict[k] += (v * frac).to(v.dtype)

            self.gen.load_state_dict(gen_aggr_dict)
            self.discr.load_state_dict(discr_aggr_dict)

            loss_sum_gen = sum(
                self.clients[c_i].round_loss_gen for c_i in indices_chosen_clients)
            loss_sum_discr = sum(
                self.clients[c_i].round_loss_discr for c_i in indices_chosen_clients)
            pbar.set_postfix(loss_sum_gen=loss_sum_gen, loss_sum_discr=loss_sum_discr)

    def sample(
            self, nr_samples: int, sample_seed=0) -> torch.Tensor:
        print(f"Generating {nr_samples} images...")
        self.gen.eval()
        torch.manual_seed(sample_seed)

        with torch.no_grad():
            fake = self.gen(torch.randn(nr_samples, self.gen.nz, 1, 1, device=DEVICE))
            fake = fake.detach().cpu()

        return fake

Mimic centralized GAN training:

In [ ]:
server = GanFedAvgServer(
    lr=2e-4,
    batch_size=16,
    client_subsets=[TRAIN_DATA],
    client_fraction=1,
    nr_local_epochs=50,
    beta1=0.5,
    seed=0,
)
server.run(nr_rounds=1)
_ = plot(server.sample(16))

Train federated GAN:

In [ ]:
server = GanFedAvgServer(
    lr=2e-4,
    batch_size=16,
    client_subsets=CLIENT_SPLIT,
    client_fraction=0.5,
    nr_local_epochs=1,
    beta1=0.5,
    seed=0,
)
server.run(nr_rounds=100)
_ = plot(server.sample(16))